# 뉴스로 주가를 예측할 수 있을까? — AI 논문 하나를 끝까지 검증해 보는 실습

이 노트북은 **"AI가 뉴스를 읽고 주가를 예측한다"**는 2015년 논문(Ding et al., IJCAI)을
실제 데이터로 재현하고, 그 주장이 **진짜인지 끝까지 의심해 보는** 수업입니다.

> 대상: 프로그래밍/통계를 깊게 배우지 않은 1학년.
> 코드는 이해하지 못해도 됩니다. **셀을 위에서부터 실행하고, 출력과 그림을 "읽는 법"을 배우는 것**이 목표입니다.

## 이 실습의 3부 구성 (하나의 이야기입니다)

| 부 | 질문 | 결말 미리보기 |
|---|---|---|
| **1부. 논문 따라하기** | 논문 방법을 그대로 만들면 정말 예측이 될까? | 만들어지긴 하는데... 돈은 안 벌린다 |
| **2부. 탐정 수사** | 왜 안 될까? 모델이 나빠서? 질문이 나빠서? | "내일 오를까?"라는 질문 자체가 문제였다 |
| **3부. 질문 바꾸기** | 질문을 바꾸면(한국 주식, "오늘 +5% 터질까?") 될까? | 맞히는 건 되는데, 돈 버는 건 또 다른 문제 |

## 실행 전 딱 3가지만

1. **커널**: README의 Setup 순서대로 만든 가상환경(`python -m venv .venv` + `pip install -r requirements.txt`)의 Jupyter 커널을 선택하세요.
2. 몇 시간 걸리는 원래 학습은 **다시 돌리지 않습니다.** 이미 학습해 둔 결과물(`artifacts/` 폴더)과
   **저장된 AI 두뇌 파일(`ntn.pt`)**을 불러와서 씁니다. (8번 셀에서만 1분짜리 미니 학습을 직접 돌려봅니다.)
3. 그림 그리는 코드는 전부 `src/dlfe_lab/viz.py` 파일에 있고, 노트북은 **불러 쓰기만** 합니다.

## 미니 용어 사전 (이것만 알면 됩니다)

- **임베딩(embedding)** — 단어나 문장을 **숫자 좌표(지도 위의 점)**로 바꾼 것. 뜻이 비슷하면 지도에서 가까이 있습니다.
- **모델(model)** — 데이터를 먹고 예측을 뱉는 **수식 기계**. 안에 조절 나사(파라미터)가 수만~수백만 개 있습니다.
- **학습(training)** — 과거 데이터로 그 나사들을 조금씩 돌려 맞추는 과정.
- **정확도(accuracy)** — 100번 중 몇 번 맞혔나. **함정**: 무조건 "오른다"라고만 답해도 53.6% 맞습니다(오른 날이 더 많아서).
- **MCC** — "찍기"를 0점으로 놓는 공정한 점수. 0 = 찍기와 같음, 1 = 완벽. 정확도의 함정을 잡아 줍니다.
- **백테스트(backtest)** — 과거로 돌아가 "이 전략대로 투자했으면 **돈을 벌었을까?**"를 계산해 보는 것.

In [ ]:
# [준비운동] 도구 상자를 불러옵니다. 이 셀은 그냥 실행만 하면 됩니다.
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib
try:
    get_ipython()            # Jupyter라면 그림이 화면에 바로 뜹니다
except NameError:
    matplotlib.use("Agg")    # (스크립트로 돌릴 때만) 화면 없이 그림

import numpy as np
import pandas as pd

from dlfe_lab import paths, data, embeddings, modeling, backtest, kr, viz
paths.bootstrap()

print("프로젝트 폴더 :", paths.ROOT)
print("결과물 폴더   :", paths.ART)
print("연구 코드 폴더:", [p.name for p in paths.FLOW_DIRS])

✅ 위 셀에서 오류 없이 폴더 3개(`flow1...`, `flow2...`, `flow3...`)가 보이면 준비 끝입니다.

---
# 1부. 논문 따라하기 — "뉴스 → 사건 → 숫자 → 예측"

논문의 아이디어는 단순합니다.

> 뉴스 제목에서 **"누가 - 무엇을 - 했다"**라는 사건을 뽑고, 그 사건을 숫자로 바꿔서,
> **"내일 주가가 오를까 내릴까"**를 맞히는 AI를 만들자.

우리는 엔비디아(NVDA) 한 종목으로 이걸 그대로 만들었습니다. 지금부터 그 재료들을 하나씩 열어 봅니다.

## 1-1. 재료 ① 뉴스 — 우리가 다루는 뉴스는 어떤 데이터일까?

아래 셀은 수집해 둔 **뉴스 제목 87,254개**를 열어서 보여줍니다.
- 어떤 회사 뉴스가 몇 개인지
- 언제부터 언제까지인지
- 실제 제목 5개 (눈으로 직접 읽어보세요!)

In [ ]:
news = data.load_news()
st = data.news_stats(news)
print(f"기사 수      : {st['rows']:,}")
print(f"티커 수      : {st['tickers']}  기간: {st['date_min']} ~ {st['date_max']}")
print(f"토큰 길이    : mean {st['tokens_mean']:.1f} / median {st['tokens_p50']:.0f} / max {st['tokens_max']}")
print(f"임베딩 학습용(pre-test) 비율: {st['emb_eligible_ratio']:.1%}")
print("\n티커별 기사 수:")
print(st["per_ticker"].to_string())
print("\nNVDA 실제 헤드라인 5개:")
for t in news[news.ticker == "NVDA"].title_clean.head(5):
    print("  -", t)
viz.news_overview(news)

### 📖 방금 본 것 읽는 법

- **왼쪽 그림**: 달마다 기사가 몇 개 나왔는지. 선이 치솟는 달 = 그 회사에 큰일이 있었던 달.
- **가운데 그림**: 회사별 기사 수. **NVDA가 2만 건**으로 뉴스의 주인공입니다.
- **오른쪽 그림**: 제목 하나가 평균 **10단어** 정도라는 뜻. AI는 이 짧은 문장만 보고 판단해야 합니다.

**직접 읽어본 헤드라인 5개, 어땠나요?** 절반이 `Law Firm`(로펌)의 소송 공지입니다.
실제 금융 뉴스 데이터는 이렇게 **광고·공지 같은 잡음이 아주 많습니다.**
"AI에게 뉴스를 읽힌다"는 말의 현실은, 이런 잡음 더미에서 진짜 정보를 골라내는 일입니다.

## 1-2. 재료 ② 주가 — 그래서 뭘 맞히는 시험인가?

시험 문제: **"내일 종가가 오늘보다 오를까, 내릴까?"** (O/X 문제)

데이터를 시간 순서로 세 구간으로 나눕니다. 공부는 과거로만, 시험은 미래로만 — **미래를 미리 보는 반칙(커닝)을 막기 위해서**입니다.

| 구간 | 비유 | 기간 |
|---|---|---|
| train | 교과서 (공부용) | 2018~2023 |
| dev | 모의고사 (연습용) | 2024 |
| test | 수능 (진짜 시험) | 2025~2026.6 |

In [ ]:
prices = data.load_prices()
ps = data.price_stats(prices)
for split in ("train", "dev", "test"):
    s = ps[split]
    print(f"{split:5s}: {s['days']:4d}일  {s['start']} ~ {s['end']}  상승비율 {s['up_rate']:.3f}")
viz.price_overview(prices)

### 📖 방금 본 것 읽는 법

- **왼쪽 그림**: NVDA 주가. 색깔 배경이 공부용(파랑)/연습용(노랑)/시험용(빨강) 구간입니다.
- **오른쪽 그림**: 하루 수익률의 분포. 0 근처가 제일 많고, 가끔 ±10% 넘게 튀는 날도 있습니다.

⭐ **이 숫자 하나는 꼭 기억하세요: 시험 구간 상승비율 = 0.536.**
아무 생각 없이 매일 "오른다"라고만 답해도 **53.6%는 맞는다**는 뜻입니다.
그래서 이 실습에서 "정확도 54%"는 자랑이 아닙니다. **찍기(53.6%)보다 얼마나 더 맞혔는지**가 진짜 실력입니다.

## 1-3. 뉴스 문장을 "사건 카드"로 바꾸기

AI는 문장을 통째로 이해하지 못하니, 논문은 제목을 **(누가 | 무엇을 했다 | 무엇에게)** 세 칸짜리 카드로 요약합니다.

> "Nvidia beats earnings expectations" → **(nvidia | beats | earnings expectations)**

아래 셀에서 실제로 만들어진 카드 44,130장 중 NVDA 카드 8장을 꺼내 봅니다.

In [ ]:
ev = data.load_events()
ok = data.load_event_ok()
es = data.event_stats(ev, ok)
print(f"이벤트 수: {es['rows']:,}  | in-vocab {es['in_vocab']:,} ({es['in_vocab_ratio']:.1%})")
print("\n티커별 이벤트 수:")
print(es["per_ticker"].to_string())
print("\nNVDA SVO 이벤트 8개:")
nv_ok = np.where(ok & (ev.ticker.values == "NVDA"))[0]
for i in nv_ok[:8]:
    print(f"  {pd.Timestamp(ev.date.iloc[int(i)]).date()}  {data.triple_str(ev, int(i))}")

### 📖 방금 본 것 읽는 법

- 카드 형식은 `(누가 | 했다 | 무엇을)` 입니다. 예: `(schall law firm | announce | filing)` = "샬 로펌이 | 발표했다 | 소장 제출을".
- `in-vocab 91%`는 "카드의 단어들이 AI 사전에 등록된 비율"입니다. 사전에 없는 단어가 든 카드 9%는 버려집니다.
- 여기서도 로펌 카드가 줄줄이 보이죠? **1-1에서 본 잡음이 카드에도 그대로 흘러들어온다**는 뜻입니다.

## 1-4. 단어를 지도 위의 점으로 — 임베딩과 PCA

이제 단어 7,024개를 각각 **100개의 숫자(=100차원 좌표)**로 바꿉니다. 이것이 **임베딩**입니다.
비슷한 문장에서 자주 같이 나온 단어일수록 좌표가 가까워집니다.

문제는 100차원 지도를 사람이 볼 수 없다는 것. 그래서 **PCA**라는 기법으로
**"100차원 지도를 2차원 종이에 최대한 덜 구겨지게 눌러서 그립니다."**
지구본(3차원)을 세계지도(2차원)로 펴는 것과 같은 일입니다.

> PCA 그림에서 볼 것은 딱 하나: **"어떤 점들이 서로 모여 있는가?"**
> 가까이 모인 점 = 컴퓨터가 "비슷한 뜻"이라고 배운 단어들.

In [ ]:
vocab, w2i, W = embeddings.load_word_vectors()
print("vocab:", len(vocab), "| dim:", W.shape[1])
for q in ("nvidia", "soars", "falls"):
    try:
        nb = embeddings.nearest_words(q, k=6)
        print(f"\n'{q}' 이웃: " + ", ".join(f"{w}({c:+.2f})" for w, c in nb))
    except KeyError as e:
        print(f"\n{e}")
xy = embeddings.pca_2d(W[:400])
labels = [vocab[i] if i < 30 else None for i in range(400)]
viz.embedding_scatter(xy, labels=labels, title="단어 지도 (자주 나온 400단어를 2차원 종이에 펼침)")

### 📖 방금 본 것 읽는 법

- `'nvidia' 이웃: amd, nvda, intel...` — AI가 스스로 **"이건 반도체 회사 동네"**라고 배웠습니다. 아무도 안 가르쳐 줬는데요!
- 괄호 안 숫자는 **가까운 정도**(1에 가까울수록 이웃).
- 그런데 이상한 점: **'soars'(급등)의 이웃 1등이 'drops'(급락)**입니다. 반대말인데 왜 붙어 있을까요?
  → 두 단어 모두 **"주가가 크게 움직였다"는 똑같은 자리**에 나오기 때문입니다.
  ("Nvidia stock ___ 5%"의 빈칸에 둘 다 들어가죠.)

⚠️ 이게 이 수업 최대의 복선입니다: **AI는 "큰일이 났다"는 것은 잘 배우지만,
"오를 큰일인지 내릴 큰일인지" 구분은 서툽니다.** 이 약점이 뒤에서 계속 발목을 잡습니다.

## 1-5. 사건 카드를 압축하는 기계 — NTN, 그리고 "저장된 두뇌" 불러오기

**NTN**은 카드의 세 칸(누가/했다/무엇을)을 받아 **사건 전체를 숫자 100개로 압축하는 기계**입니다.
안에는 조절 나사(파라미터)가 **306만 개** 있습니다.

여기서 중요한 것: 우리는 이 나사를 지금 돌리지 않습니다.
**예전에 몇 시간 들여 학습시킨 나사 설정값이 `artifacts/ntn.pt` 파일에 저장**되어 있고,
아래 셀은 그 파일을 불러올 뿐입니다. — 게임의 "세이브 파일 로드"와 같습니다.

In [ ]:
ntn = embeddings.load_ntn()          # 저장된 두뇌(ntn.pt)를 불러옵니다. 학습 안 함!
print(modeling.describe_model(ntn))

### 📖 방금 본 것 읽는 법

- `T1, T2, T3, W1...`은 기계 내부 부품 이름입니다. 외울 필요 없습니다.
- 마지막 줄 **`total params: 3,060,400`** — 이 기계에 조절 나사가 306만 개 있고,
  전부 **예전 학습에서 이미 맞춰진 값**으로 세팅됐다는 뜻입니다.

## 1-6. 저장된 두뇌로 직접 "추론"해 보기 — 임베딩 벡터 구경

이제 불러온 NTN에 사건 카드 3장을 실제로 넣어서 **숫자 100개짜리 벡터**가 나오는 걸 직접 봅니다.

미리 알아 둘 것 두 가지:
1. 기계의 원래 출력(raw)은 거의 **+1 아니면 -1**입니다. 학습이 끝나면 내부 스위치들이
   확실하게 켜지거나(+1) 꺼지는(-1) 상태로 굳기 때문입니다. 그래서 **306만 개 나사의 결과물이
   사실상 100자리 모스부호(±1 코드)**처럼 보입니다.
2. 실제 파이프라인은 이 코드를 **표준화**(평균 0 기준으로 재조정)한 버전을 씁니다. 아래에서 둘 다 보여줍니다.

**벡터끼리 비교하는 법**: `cosine`(코사인) 점수 하나만 기억하세요.
**+1 = 같은 방향(비슷한 사건), 0 = 상관없음, -1 = 반대 방향.**

In [ ]:
# 소송 공시 스팸(announce/remind)이 아닌 NVDA 이벤트를 골라야 이웃 탐색이 유의미해집니다.
cand = [int(i) for i in nv_ok if " ".join(ev.p.iloc[int(i)]) not in ("announce", "remind")]
pick = cand[:3]

U = embeddings.embed_events(ntn, ev, pick)     # 저장된 NTN 두뇌로 직접 추론!
emb = embeddings.load_event_embeddings()       # 파이프라인용 표준화 임베딩 (44,130 x 100)
print(f"raw NTN 출력 범위: [{U.min():+.3f}, {U.max():+.3f}]  <- 거의 ±1 (스위치처럼 굳음)")

ok_rows = np.where(ok)[0]
var_dims = np.argsort(emb[ok_rows].var(0))[::-1][:8]
print("사건들을 가장 잘 구분하는 8개 자리(차원):", var_dims.tolist())
for j, i in enumerate(pick):
    print("\n" + data.triple_str(ev, i))
    print("   raw 출력(±1 코드)  =", np.round(U[j, var_dims], 3))
    print("   표준화 벡터        =", np.round(emb[i, var_dims], 3))

En3 = emb[pick] / (np.linalg.norm(emb[pick], axis=1, keepdims=True) + 1e-9)
print("\n세 사건끼리의 cosine 점수표 (대각선=자기 자신=1):")
print(np.round(En3 @ En3.T, 3))

print("\n첫 사건과 가장 비슷한 사건 TOP5 (44,130장 중, 중복 제거):")
for idx_, cos_, trip in embeddings.nearest_events(ev, emb, pick[0], k=5):
    print(f"   {cos_:+.3f}  {trip}")

rng = np.random.default_rng(13)
sample = rng.choice(np.where(ok)[0], 4000, replace=False)
xy_ev = embeddings.pca_2d(emb[sample])
viz.embedding_scatter(xy_ev, highlight_mask=(ev.ticker.values[sample] == "NVDA"),
                      title="사건 지도 (4,000개 사건, 빨간 점 = NVDA 사건)")

### 📖 방금 본 것 읽는 법

1. **±1 코드**: 세 사건의 raw 출력이 똑같아 보여도 놀라지 마세요. 100자리 중 이 8자리가 우연히 같은 것이고,
   바로 아래 **cosine 점수표**를 보면 셋이 서로 다른 사건임이 드러납니다
   (예: -0.19 = 오히려 살짝 반대 방향).
2. **TOP5 이웃**: 첫 사건 `(it | investigate | claims)`(조사하다)의 이웃에
   `(visa | investigate | ...)`, `(pomerantz law firm | investigate | claims)` 처럼
   **다른 회사의 '조사' 사건들**이 모였습니다. → 기계가 "사건의 종류"를 이해했다는 증거입니다.
3. **사건 지도(PCA)**: 빨간 점(NVDA 사건)이 한 구석에 몰려 있지 않고 회색(다른 회사들) 사이에 섞여 있죠.
   → 사건의 좌표는 "어느 회사냐"가 아니라 **"무슨 종류의 일이냐"**로 정해진다는 뜻입니다.

## 1-7. 예측 기계(CNN) 구경 — 이제 카드로 시험을 치는 기계

사건 벡터가 준비됐으니, 마지막으로 **"내일 오를까?"에 답하는 기계**를 봅니다.

구조를 비유하면: 이 기계는 시험 볼 때
**최근 1일(어제), 최근 1주, 최근 1달치 사건 카드 묶음** 세 개를 책상에 놓고,
1달/1주 묶음은 **돋보기(CNN)로 3장씩 훑으며 중요한 부분만 골라낸 뒤**, 셋을 합쳐 O/X를 찍습니다.

In [ ]:
from s6_models import DenseModel      # 논문 재현에 실제로 쓰인 진짜 모델 클래스입니다
cnn = DenseModel(100, nn_only=False)
print(modeling.describe_model(cnn))

### 📖 방금 본 것 읽는 법

- `Conv1d(...)` 두 줄이 바로 그 "돋보기"입니다 (1달치용, 1주치용).
- 나사 수는 **61,529개** — NTN(306만 개)보다 훨씬 작은 기계입니다.
  큰 기계(재료 가공) + 작은 기계(최종 판단)의 조합이죠.

## 1-8. 학습을 "잠깐만" 직접 돌려보기 + 성적 그래프 읽기

이번엔 구경만 하지 말고, **진짜 학습을 6바퀴만** 돌려 봅니다 (1분 이내).
"바퀴(epoch)" = 교과서(train 구간)를 처음부터 끝까지 한 번 다 푸는 것.

그래프에서 볼 두 개의 선:
- **파란 선 (train loss)** = 교과서 문제에서 틀리는 정도. **내려가면 공부가 되고 있는 것.**
- **빨간 선 (dev MCC)** = 모의고사 실력 점수. **0이면 찍기와 같은 수준.**

In [ ]:
hist = modeling.quick_train("EB", nn_only=True, epochs=6)
for i, (l, m) in enumerate(zip(hist["train_loss"], hist["dev_mcc"]), 1):
    print(f"epoch {i}: train BCE {l:.4f} | dev MCC {m:+.4f}")
print(f"\n(6바퀴짜리 미니 학습의 시험 성적) test acc {hist['test_acc']:.4f} | test MCC {hist['test_mcc']:+.4f}")
viz.training_curves(hist)

### 📖 방금 본 것 읽는 법

- 파란 선이 0.704 → 0.694로 **아주 조금** 내려갔습니다. 공부가 되긴 되는데 몹시 더딥니다.
- 빨간 선(모의고사 MCC)은 0 언저리에서 오르락내리락 — **아직 찍기 수준**이라는 뜻입니다.
- 시험 정확도 0.5355 = 정확히 "무조건 오른다 찍기"와 같은 점수. 6바퀴로는 아무것도 못 배운 겁니다.

> 그럼 제대로 공부시키면? — 원래 연구에서는 **120바퀴 + 조기중단 + 시드 4개 앙상블**로
> 몇 시간 학습을 했고, 그 성적표가 이미 저장돼 있습니다. 다음 셀에서 열어 봅니다.

## 1-9. 제대로 학습시킨 성적표 열기 — "정확도의 함정" 체험

`results.json` = 논문의 모델 7종 + 업그레이드 4종을 **풀코스로 학습시킨 최종 성적표**입니다.

표 읽는 법:
- `test_acc`(시험 정확도)에서 **0.536(찍기)**을 빼고 보세요. 그게 진짜 실력입니다.
- `test_mcc`는 처음부터 찍기=0으로 보정된 점수입니다. **0.1도 안 되면 "거의 찍기"**입니다.

In [ ]:
res = json.load(open(paths.ART / "results.json", encoding="utf-8"))
tbl = pd.DataFrame(res["metrics"]).set_index("model").round(4)
print(tbl.to_string())
print(f"\nn_test={res['n_test']}  majority={res['test_up_rate']:.4f}")
viz.results_bar(res)

### 📖 방금 본 것 읽는 법

- 제일 잘한 모델(TEB-CNN)이 **55.5%** — 찍기(53.6%)보다 **고작 +1.9%p** 높습니다.
- MCC로 봐도 최고 **+0.08** — "찍기보다 아주 조금 나은 수준"입니다.
- 그래도 논문의 주장(사건 임베딩 EB가 단어 임베딩 WB보다 낫다, CNN이 낫다)의 **순위 자체는 재현**됐습니다.

**한 줄 정리: 논문은 거짓말은 아니었다. 하지만 효과의 크기가 아주 작다.**
그럼 이 작은 실력으로... 돈은 벌 수 있을까요? 다음이 이 실습의 첫 백테스트입니다.

## 1-10. 백테스트 — "이 전략대로 투자했으면 돈을 벌었을까?"

**전략 (논문에 나온 그대로):**
> 매일 아침, AI가 "오른다"고 하면 **$10,000어치 매수** — 장중 +2% 도달하면 즉시 팔고, 아니면 종가에 판다.
> "내린다"고 하면 반대로 공매도(내리면 이익) — -1% 도달하면 정리.

이걸 시험 구간 **366일 동안 매일** 반복했으면 얼마가 남았을까요?
사용하는 예측 확률은 **예전에 학습해 둔 최종 모델(TEB-CNN 앙상블)이 실제로 출력했던 값**입니다.

그리고 마지막 그림이 중요합니다.
**"동전 던지기로 사고판 가짜 투자자 1,000명"**을 시뮬레이션해서 우리 성적과 비교합니다.
우리가 실력자라면, 1,000명 대부분을 이겨야 정상이겠죠?

In [ ]:
teb = backtest.load_teb_daily()
print(teb.head(3).to_string(index=False))
sim = backtest.paper_simulate(teb.prob_up.values)
print(f"\nalways-trade: 총손익 ${sim['total']:,.0f} / {sim['n_trades']}일")
print(f"(교차검증: CSV 저장 손익 합 = ${teb.profit_always.sum():,.0f})")
viz.equity_curve(sim["dates"], sim["daily"], title="매일 $10,000 베팅했을 때 내 잔고의 여정 (18개월)")
dist, p = backtest.randomization(sim["total"], n=1000)
viz.randomization_hist(dist, sim["total"], p)
sim70 = backtest.paper_simulate(teb.prob_up.values, threshold=0.70)
print(f"자신 있을 때(확률 70% 이상)만 거래: 총손익 ${sim70['total']:,.0f} / 거래 {sim70['n_trades']}회")

### 📖 그래서, 돈을 벌었을까?

- **총손익 +$202.** 18개월 동안 매일 $10,000(약 1,400만 원)씩 걸어서 번 돈이 **커피 몇 잔 값**입니다.
- **잔고 여정 그래프**: 중간에 +$4,000 근처까지 갔다가 다 반납합니다. 0선 위아래를 표류하는 모양 —
  이런 곡선은 "실력"이 아니라 "출렁임"의 모양입니다.
- **동전 던지기 1,000명과의 비교**: 우리보다 잘 번 가짜 투자자가 **366명(p=0.366)**.
  실력자라면 이 숫자가 50명(p=0.05) 아래여야 합니다. → **운과 구분이 안 됩니다.**
- 심지어 "자신 있을 때만 거래" 전략은 **-$605 손실**. 자신감도 믿을 게 못 됩니다.

**1부 결론: 논문 재현 성공 ✔ / 돈 벌기 실패 ✘.**
왜 실패했는지 — 2부에서 범인을 찾으러 갑니다.

---
# 2부. 탐정 수사 — 왜 안 될까? 모델 탓? 질문 탓?

용의자는 둘입니다.
- 용의자 A: **모델이 후져서** (더 좋은 AI를 쓰면 해결)
- 용의자 B: **"내일 오를까?"라는 질문 자체가 나빠서** (무엇을 써도 해결 불가)

실제 연구에서 업그레이드를 총동원해 조사했습니다. 그 기록을 열어 봅니다.

## 2-1. 단서 ① 뉴스는 "예언"일까, "생중계"일까?

FinBERT라는 **금융 전문 AI**로 NVDA 뉴스 제목 21,576개에 **감성 점수**를 매겼습니다.
(좋은 소식 = +, 나쁜 소식 = -)

이제 상관계수를 봅니다. **상관계수** = 두 숫자가 같이 움직이는 정도 (0 = 남남, 1 = 완전 커플).
- 감성 vs **당일** 수익률의 상관 → 뉴스가 "오늘 일어난 일"을 담는 정도
- 감성 vs **다음날** 수익률의 상관 → 뉴스로 "내일을 예측"할 수 있는 정도 ← 우리가 원하는 건 이것!

In [ ]:
nv_news = news[news.ticker == "NVDA"].reset_index(drop=True)
sent = np.load(paths.ART / "tf_title_sent.npy")
assert len(sent) == len(nv_news), (len(sent), len(nv_news))
nv_news["sent"] = sent
daily_sent = nv_news.groupby("date")["sent"].mean()
samp = data.load_samples()
joined = samp.set_index("date").join(daily_sent.rename("sent"), how="inner")
same_day = float(joined["sent"].corr(joined["ret"]))
next_day = float(joined["sent"].corr(joined["ret"].shift(-1)))
print(f"corr(당일 감성, 당일 수익률)   = {same_day:+.3f}")
print(f"corr(당일 감성, 다음날 수익률) = {next_day:+.3f}   <- 예측에 쓸 수 있는 쪽")
viz.sentiment_vs_returns(joined["sent"].values[:-1], joined["ret"].shift(-1).dropna().values)

### 📖 방금 본 것 읽는 법

- 당일 상관 **+0.21** vs 다음날 상관 **+0.01**. 차이가 20배가 넘습니다.
- 산점도를 봐도 점들이 아무 방향성 없이 **구름처럼 퍼져** 있죠 — "오른쪽으로 갈수록 위로" 같은 기울기가 없습니다.

**해석: 뉴스는 예언이 아니라 생중계입니다.**
좋은 뉴스가 나올 때는 주가가 **이미 그날** 올라 있고, 우리가 그걸 읽고 다음날 사면 늦습니다.
(밤사이 뉴스조차 다음날 **개장가에 이미 반영**된 채 시작합니다.)

## 2-2. 단서 ② 업그레이드를 다 부어 봤다 — "정확도의 천장"

연구에서 실제로 시도한 업그레이드들: 주가 기술지표 추가, 자신 있는 날만 예측,
관련 종목 정보 추가, 모델 크기 키우기, 그리고 **walk-forward**
(= 매 분기 다시 학습하며 한 발씩 전진하는, 커닝이 원천 불가능한 가장 정직한 시험 방식).

그 성적들을 한 그래프에 모았습니다.

In [ ]:
def _load(name):
    return json.load(open(paths.ART / name, encoding="utf-8"))

boost = _load("results_boost.json")
sel = _load("results_selective.json")
wf = _load("results_walkforward.json")
cap = _load("results_capacity.json")

ladder = {
    "s7 논문 매트릭스 best": max(r["test_acc"] for r in res["metrics"]),
    "s10 가격+뉴스 GBM best": float(boost["best_full_test_acc"]),
    "s11 selective (cov 0.5)": float(next(r["test_acc"] for r in sel["dev_quantile_rule"]
                                          if r["model"] == "price+news" and r["target_cov"] == 0.5)),
    "s13 capacity best": max(r["test"] for r in cap),
    "s15 walk-forward OOS": float(wf["oos_full_acc"]),
}
for k_, v_ in ladder.items():
    print(f"{k_:26s} {v_:.4f}")
viz.upgrade_ladder(ladder, baseline=res["test_up_rate"])

### 📖 방금 본 것 읽는 법

- 모든 막대가 점선("무조건 오른다 찍기" = 0.536) **바로 근처**에 붙어 있습니다.
- 최고 기록도 0.56 — 찍기 대비 **+2%p 남짓의 벽**을 무엇으로도 못 넘었습니다.
- 특히 가장 정직한 방식인 **walk-forward는 0.51** — 오히려 찍기보다 낮습니다.
  (한 번의 운 좋은 시험 구간이 아니라 5년 내내 시험을 보면 이렇게 됩니다.)

**용의자 A(모델 탓)의 혐의가 옅어지고 있습니다.** 뭘 갈아 끼워도 천장이 같으니까요.

## 2-3. 단서 ③ 그 "+$1,260 수익"은 진짜였을까? — 운을 실력과 구분하기

사실 연구 초기에 TEB-CNN 모델이 **+$1,260을 벌었다는 결과**가 나와 잠깐 흥분했었습니다.
그런데 이런 검증을 해 봤습니다:

> **같은 모델을 "주사위 초기값(시드)"만 바꿔서 4번 학습**해 보자.
> 진짜 실력이라면 4번 모두 비슷하게 벌어야 한다.

그리고 "여러 개 중 최고만 골라 보는" 착시를 잡는 **다중검정 보정**(Bonferroni)도 했습니다.
반 학생 11명에게 동전을 10번씩 던지게 하면 누군가는 8번 맞힙니다 — 그 아이가 초능력자가 아니듯,
**모델 11개 중 1등의 성적은 그 자체로는 증거가 아닙니다.**

In [ ]:
sv = kr.load_survival()
print("시드(주사위 초기값)별 총손익 — 같은 모델, 같은 데이터, 초기값만 다름:")
for r in sv["seed_rows"]:
    print(f"  seed {r['seed']}: dev MCC {r['dev_mcc']:+.3f} | test profit ${r['test_profit_always']:,.0f}")
ens, al, rd, mt = sv["ensemble_metrics"], sv["always"], sv["randomization"], sv["multiple_testing"]
print(f"\n4개 두뇌의 평균(앙상블): test acc {ens['test_acc']:.4f} | MCC {ens['test_mcc']:+.4f}")
print(f"always-trade 손익 ${al['profit_total']:,.0f} | 운의 범위(CI95) "
      f"[{al['bootstrap_profit_ci95'][0]:,.0f}, {al['bootstrap_profit_ci95'][2]:,.0f}] | "
      f"손해봤을 확률 P(<=0)={al['bootstrap_prob_profit_le_0']:.3f}")
print(f"동전던지기 비교 p = {rd['p_ge_always']:.3f} | 다중검정 보정(Bonferroni) p = {mt['teb_bonferroni_11_models']:.2f}")
dev_c = pd.read_csv(paths.ART / "s53_teb_dev_threshold_curve.csv")
test_c = pd.read_csv(paths.ART / "s53_teb_test_threshold_curve.csv")
viz.threshold_curves(dev_c, test_c)

### 📖 방금 본 것 읽는 법

- 시드별 손익: **+$3,563 / +$262 / -$926 / +$1,260.** 같은 모델인데 주사위 값에 따라
  **이익↔손실이 뒤집힙니다.** "+$1,260"은 실력이 아니라 **4번 중 운 좋은 1번**이었던 겁니다.
- 4개를 평균 내면 +$202 (1부 백테스트의 그 숫자!). 운의 범위는 **-$7,109 ~ +$7,230** —
  +$202는 그 한가운데 점 하나일 뿐입니다.
- 마지막 그래프: 파란 선(연습지 dev)으로 "확신 문턱값"을 고르고 → 빨간 선(실전 test)에서 확인하니
  좋아 보였던 문턱값들이 실전에서 무너집니다.
  **좋은 결과가 필요할 때까지 이것저것 골라 보는 행위 자체가 착시를 만듭니다.**

**2부 판결: 범인은 용의자 B — "내일 오를까?"라는 질문 그 자체.**
뉴스가 품은 정보는 "방향"이 아니라 "출렁임(변동성)"입니다. 그렇다면... **질문을 바꾸면 되지 않을까?**

---
# 3부. 질문 바꾸기 — 한국 주식, "오늘 장중에 +5% 터질까?"

새 질문: ~~내일 종가가 오를까?~~ →
**"오늘 아침 시가 대비, 장중에 +5%를 한 번이라도 찍을까?"**

이 질문이 더 나은 이유:
- 답이 **"큰 출렁임이 있을까?"**에 가깝습니다 — 뉴스가 실제로 잘 담고 있는 바로 그 정보!
- 아침에 예측하고 아침에 사면 되니 **타이밍이 어긋나지 않습니다.**

무대도 한국으로 옮겼습니다: **626개 종목 × 11년(2015~2026) = 164만 종목-일**의 실제 데이터입니다.

## 3-1. 새 질문의 난이도 먼저 확인 — 기본 발생률

In [ ]:
ko = kr.load_kr_ohlcv()
print(f"KR OHLCV: {len(ko):,}행 | {ko.ticker.nunique()}종목 | "
      f"{ko.date.min().date()} ~ {ko.date.max().date()}")
rates = kr.exceedance_rates(ko)
for k_, v_ in rates.items():
    print(f"  {k_}: {v_:.3%}")
viz.exceedance_bars(rates)

### 📖 방금 본 것 읽는 법

- **UP5% = 7.8%**: 아무 종목이나 찍었을 때, 그날 장중 +5%를 터치할 확률은 **13일에 한 번꼴**입니다.
- 즉 새 질문은 "흔한 O/X"가 아니라 **"드문 대박날을 미리 골라내기"** 게임입니다.
- 이 7.8%가 3-2의 기준선입니다. AI가 이보다 얼마나 잘 고르는지 보세요.

## 3-2. AI의 성적 — 아무거나 vs AI의 1순위 픽

**GBM**이라는 AI(스무고개 질문 나무 수백 그루의 다수결)를 **2021년 이전 데이터로만** 학습시키고,
그 이후 **5년치(75만 종목-일)**에 대해 매일 "오늘 +5% 터질 확률" 점수를 매기게 했습니다.
(이 점수도 예전에 학습시킨 모델이 실제로 출력해 저장해 둔 것입니다.)

매일 **점수 1등 종목 하나만** 골랐다면, 실제로 +5%를 터치한 날이 몇 %였을까요?

In [ ]:
sc = kr.load_kr_scores()
print(f"scores: {len(sc):,}행 | {sc.date.min().date()} ~ {sc.date.max().date()}")
top1 = kr.daily_topk_hit(sc, k=1)
top3 = kr.daily_topk_hit(sc, k=3)
for name, t in (("top-1", top1), ("top-3", top3)):
    print(f"{name}: 적중 {t['hit_rate']:.3f} vs 아무거나 {t['base_rate']:.3f} "
          f"({t['hit_rate']/t['base_rate']:.1f}배, {t['n_days']:,}일 동안)")
viz.score_lift({"매일 1등 픽": top1, "매일 1~3등 픽": top3})

### 📖 방금 본 것 읽는 법

- 아무거나: **7.4%** → AI 1등 픽: **54.4%**. **7배가 넘는 적중률**이고, 5년 내내(1,236일) 유지됩니다.
- 이건 2부에서 본 "찍기 +2%p"와는 차원이 다른, **통계적으로 진짜인 실력**입니다.
- 질문을 바꾸니 AI가 드디어 제 실력을 냅니다. **"내일 방향"은 못 맞혀도 "오늘 출렁일 종목"은 잘 찾습니다.**

그럼 이제 진짜 마지막 관문입니다. **이 적중률로 실제로 돈을 벌 수 있을까?**

## 3-3. 백테스트 3종 — "적중률 54%면 부자 되는 거 아닌가요?"

세 가지 전략을 각각 과거 데이터로 실험했습니다 (모두 예전에 실행해 저장된 결과입니다):

1. **전략 1**: 매일 AI 픽을 아침에 사고 → +5% 되면 팔고, 손실이 커지면 손절. (5분 단위 정밀 검증)
2. **전략 2**: "아침 말고 눌렸을 때 사면?" 등 **매수 타이밍 6가지**를 다 실험해 최고만 채택.
3. **전략 3**: 손절 없이 +5% 지정가 매도만. (하루 가격 경로 가정이 전혀 필요 없는 가장 정확한 검증)

그래프의 선 = **내 잔고 배수**입니다. 1.0에서 시작해서 **위로 가면 번 것, 아래로 가면 잃은 것.**

In [ ]:
cur = kr.load_equity_curves()
for name, df_ in cur.items():
    print(f"{name}: 시작 {float(df_.equity.iloc[0]):.3f} -> 최종 {float(df_.equity.iloc[-1]):.3f}")
viz.kr_equity_panels(cur)

### 📖 그래서, 돈을 벌었을까? — 세 번 다 아니오

- **전략 1**: 잔고 1.00 → **0.85** (100만 원이 85만 원). 적중해도 수수료와 손절이 갉아먹습니다.
- **전략 2**: 타이밍 6가지 중 **최고**를 골랐는데도 → **0.85.** 진입 시점의 문제가 아니었습니다.
- **전략 3(가장 정확)**: 0.95 → **0.24** (100만 원이 24만 원!). 손절이 없으니 못 오른 날의 하락을 그대로 다 맞습니다.

**왜 적중률 54%로도 잃을까?** 핵심은 이것입니다:

> AI가 고르는 종목은 **누가 봐도 좋은 뉴스가 난 종목**입니다.
> 그래서 아침 개장가가 **이미 비싸게 시작**합니다(갭업).
> +5%를 찍어도 내 매수가 기준으론 남는 게 없고, 못 찍은 날은 크게 빠집니다.
> **"좋은 종목을 아는 것"과 "좋은 가격에 사는 것"은 완전히 다른 문제입니다.**

## 3-4. 마지막 궁금증 — AI는 대체 "무슨 뉴스를 보고" 판단했을까? (XAI)

AI의 판단 근거를 거꾸로 추적하는 기법(**XAI**, 설명가능 AI)으로,
갭 예측 모델이 **가장 크게 맞힌 날 / 가장 크게 틀린 날**에 어떤 뉴스가 결정적이었는지 뽑아 봅니다.

In [ ]:
g = kr.load_gap_results()
print(f"갭 실험: 링크 {g['n_links']:,} | 샘플 {g['n_samples']:,} | test gap-up base {g['test_gap_up_rate']:.3f}")
rows = []
for name, v in g["variants"].items():
    d = v.get("direction", {})
    rows.append({"variant": name, "tradable": v.get("tradable"),
                 "acc": d.get("acc"), "base": d.get("base"), "mcc": d.get("mcc")})
print(pd.DataFrame(rows).round(4).to_string(index=False))

x = kr.load_xai()
print(f"\nXAI encoder: {x['encoder']}")
print("\n[가장 크게 맞힌 날 — AI가 근거로 삼은 뉴스]")
for d in x["xai_best_days"][:3]:
    top = d["top_titles"][0]["title"] if d.get("top_titles") else "(없음)"
    print(f"  {d['ticker']} {d['date']} 갭 {d['gap_pct']:+.1f}% :: {top}")
print("[가장 크게 틀린 날 — AI가 근거로 삼은 뉴스]")
for d in x["xai_worst_days"][:3]:
    top = d["top_titles"][0]["title"] if d.get("top_titles") else "(없음)"
    print(f"  {d['ticker']} {d['date']} 갭 {d['gap_pct']:+.1f}% :: {top}")

### 📖 방금 본 것 읽는 법

- **맞힌 날의 근거**: "저평가 매력 부각… 실적 개선 기대", "AI 메모리 기판 수혜… 목표가↑" —
  사람이 봐도 "오를 만한" 증권사 호평 기사입니다. AI가 엉뚱한 걸 본 게 아닙니다.
- **틀린 날의 근거**: "불성실공시법인 지정"(나쁜 뉴스인데 주가는 갭상승!), 평범한 기대 기사인데 하락 —
  **뉴스만으로는 설명이 안 되는 날**들이 존재합니다.
- 위의 표에서 갭 방향 정확도도 기준선(0.544) 대비 +0.3%p 수준 — 역시 종잇장 같은 우위입니다.

이렇게 **AI의 판단 근거를 눈으로 직접 검수하는 습관**이, 점수표만 믿는 것보다 훨씬 중요합니다.

---
# 마무리 — 세 부의 이야기를 숫자로 요약

In [ ]:
summary = {
    "1부 논문 재현": {
        "최고 시험 정확도": round(max(r["test_acc"] for r in res["metrics"]), 4),
        "무조건 '오른다' 찍기": round(res["test_up_rate"], 4),
        "백테스트": f"${sim['total']:,.0f} (동전던지기 1000명 중 {int(p*1000)}명이 우리보다 잘 벎)",
    },
    "2부 US 진단": {
        "업그레이드 총동원 최고": round(max(ladder.values()), 4),
        "가장 정직한 시험(walk-forward)": round(float(wf["oos_full_acc"]), 4),
        "다중검정 보정 후 p": mt["teb_bonferroni_11_models"],
    },
    "3부 KR +5% 예측": {
        "AI 1등 픽 적중률 vs 아무거나": f"{top1['hit_rate']:.3f} vs {top1['base_rate']:.3f} (7배)",
        "그런데 백테스트": "전략 3종 모두 손실 — 좋은 종목 != 좋은 가격",
    },
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 오늘 가져갈 세 문장

1. **"재현된다"와 "쓸모 있다"는 다르다.** 논문 방법은 재현됐지만, 효과는 찍기보다 +2%p였습니다.
2. **성과가 보이면 운부터 의심하라.** 시드 4개, 동전던지기 1,000명, 다중검정 —
   이 세 가지 검증을 통과 못 하면 그 수익은 우연입니다.
3. **좋은 질문이 좋은 모델을 이긴다.** 질문을 바꾸자 적중률이 7배가 됐습니다.
   그리고 마지막 벽 — **시장은 좋은 소식을 내가 사기 전에 가격에 반영해 버린다** — 도 배웠습니다.

## 직접 해보기 (과제)

1. **8번 셀**에서 `epochs=6`을 `epochs=30`으로 바꿔 다시 실행해 보세요.
   파란 선(loss)은 더 내려가는데 빨간 선(모의고사 MCC)도 같이 좋아지나요?
   → "오래 공부한다고 실력이 느는 건 아니다"를 직접 확인해 보세요.
2. **1-10 백테스트 셀**에서 `threshold=0.70`을 0.55, 0.60, 0.80으로 바꿔 보세요.
   "더 자신 있을 때만 거래"가 정말 더 안전한가요? 거래 횟수는 몇 번으로 줄어드나요?
3. **3-2 셀**에서 `kr.daily_topk_hit(sc, k=1, tp=0.05)`의 `tp=0.05`(+5%)를
   `0.02`(+2%)로 바꿔 보세요. 목표가 쉬워지면 적중률과 "아무거나" 기준선이 각각 어떻게 변하나요?